# **NLP Practical 2 — POS Tagging and Chunking**

---

**Name: Shubham Sarwar**  
**PRN: 202301040127**

**Aim:** Identify parts of speech (POS) for each word, extract noun and verb phrases
using chunking, and visualize dependency relations using **spaCy**.


## **1. Install spaCy and Download the English Model**

In [ ]:
# spaCy = a modern, fast NLP library (alternative to NLTK).
# 'en_core_web_sm' is the SMALL pre-trained English model that contains:
#   - tokenizer
#   - POS tagger
#   - dependency parser
#   - named entity recognizer (NER)
!pip install spacy
!python -m spacy download en_core_web_sm

## **2. Import Libraries and Load the Model**

In [ ]:
import spacy                        # main spaCy library
from spacy import displacy           # displacy = built-in tool for VISUALIZING parses

# Load the English language model (must be downloaded first).
# `nlp` is now a pipeline object: we feed text into it and it does all NLP steps automatically.
nlp = spacy.load("en_core_web_sm")

## **3. Input Text**

In [ ]:
text = """Artificial Intelligence is transforming the world.
It enables computers to understand language and make decisions."""

# Pass the text through the spaCy pipeline.
# `doc` is a special spaCy object: a sequence of tokens with all linguistic info attached.
doc = nlp(text)

## **4. POS Tagging**

**POS = Part Of Speech.**  
For each word the model predicts its grammatical category:  
`NOUN, VERB, ADJ (adjective), ADV (adverb), PRON (pronoun), DET (the/a), AUX (is/was), …`

For each token we print:
- `token.text` → the word itself
- `token.pos_` → coarse POS tag (universal: NOUN, VERB…)
- `token.tag_` → fine-grained tag (NN, NNS, VBZ, VBG… Penn Treebank style)
- `token.dep_` → its **dependency relation** to the parent word (nsubj, dobj, ROOT…)


In [ ]:
print("===== POS TAGGING =====")
print(f"{'TOKEN':15} {'POS':10} {'TAG':10} DEPENDENCY")
print("-" * 50)

for token in doc:
    # f-string formatting: :15 = pad to 15 chars wide, makes the output a neat table
    print(f"{token.text:15} {token.pos_:10} {token.tag_:10} {token.dep_}")

## **5. Chunking — Noun Phrases**

**Chunking** groups words into short phrases.  
A **Noun Phrase (NP)** is a noun together with its describing words:  
`"the world"`, `"Artificial Intelligence"`, `"a smart computer"` …

spaCy gives us noun chunks for free through `doc.noun_chunks`.


In [ ]:
print("===== NOUN PHRASES (NP) =====")
for chunk in doc.noun_chunks:
    # chunk.text = the phrase, chunk.root.text = the head noun, chunk.root.dep_ = its role
    print(f"{chunk.text:30} | head: {chunk.root.text:15} | role: {chunk.root.dep_}")

## **6. Chunking — Verb Phrases**

A **Verb Phrase (VP)** is a verb plus its helpers/auxiliaries (is running, will go, has been studying).  
spaCy doesn't have a built-in `verb_chunks`, so we build one ourselves:  
for every VERB token, we collect it together with its AUX children (is, has, was…) and the PART "to".


In [ ]:
print("===== VERB PHRASES (VP) =====")

verb_phrases = []
for token in doc:
    if token.pos_ == "VERB":                 # look only at main verbs
        # children = tokens that depend on this verb (e.g., its auxiliary "is")
        aux_parts = [child.text for child in token.children
                     if child.pos_ in ("AUX", "PART")]
        # build the phrase in the order: auxiliaries first, then the main verb
        phrase = " ".join(aux_parts + [token.text])
        verb_phrases.append(phrase)

for vp in verb_phrases:
    print(vp)

## **7. Dependency Visualization**

**Dependency parsing** tells us HOW words are connected.  
Example: in *"Eleanor reads books"*, "reads" is the ROOT, "Eleanor" is its `nsubj`
(nominal subject), and "books" is its `dobj` (direct object).

`displacy` draws this as a nice arrow diagram.  
In Colab use `jupyter=True` to render inline; we also save it to an SVG file.


In [ ]:
# Render the dependency tree INSIDE the notebook (works in Jupyter / Colab)
displacy.render(doc, style="dep", jupyter=True)

In [ ]:
# Also save the SVG to disk so you can submit it as proof / put in the report.
svg = displacy.render(doc, style="dep", jupyter=False)

with open("dependency.svg", "w", encoding="utf-8") as f:
    f.write(svg)

print("Dependency tree saved as: dependency.svg")